# Semantic Segmentation using YOLO-sem or SAM3

In [ ]:
import os

import cv2


project_root = os.path.abspath(
    os.path.join(os.path.dirname(__name__))
)

cityscapes_class_to_name = {
    0: "road",
    1: "sidewalk",
    2: "building",
    3: "wall",
    4: "fence",
    5: "pole",
    6: "traffic light",
    7: "traffic sign",
    8: "vegetation",
    9: "terrain",
    10: "sky",
    11: "person",
    12: "rider",
    13: "car",
    14: "truck",
    15: "bus",
    16: "train",
    17: "motorcycle",
    18: "bicycle"
}
cityscapes_name_to_id = {v.lower(): k for k, v in cityscapes_class_to_name.items()}

In [ ]:
images_folder = "texture_output_Rozenstraat"

output_folder = os.path.join(project_root, "yolo_sem_Rozenstraat")

pred_output_folder = os.path.join(output_folder, "predictions")
mask_output_folder = os.path.join(output_folder, "masks")

os.makedirs(pred_output_folder, exist_ok=True)
os.makedirs(mask_output_folder, exist_ok=True)

In [ ]:
image_paths = [
    os.path.join(images_folder, file)
    for file in os.listdir(images_folder)
    if os.path.splitext(file)[1] in [".png", ".jpg", ".jpeg"]
]

# YOLO segmentation

In [ ]:
from ultralytics import YOLO


# Load a pretrained YOLO26n-sem Semantic model
model_path = f"{project_root}/model_weights/yolo26m-sem.pt"
model = YOLO(model_path)


for img_path in image_paths:
    print(img_path)

    # Run inference on an image
    result = model(img_path, classes=[8], imgsz=1280, conf=0.25)[0]  # result

    # Save results to disk
    img_name, img_ext = os.path.splitext(
        os.path.basename(result.path)
    )
    filename = os.path.join(
        pred_output_folder,
        f"{img_name}_prediction{img_ext}"
    )
    result.save(filename=filename)

    mask = result.semantic_mask.data == 8
    mask_img = mask.int() * 255

    mask_filename = os.path.join(
        mask_output_folder,
        f"{img_name}_mask.png"
    )
    cv2.imwrite(mask_filename, mask_img.numpy().astype("uint8"))

# SAM 3

In [ ]:
from ultralytics.models.sam import SAM3SemanticPredictor


model_path = f"{project_root}/model_weights/sam3.1_multiplex.pt"

# Initialize predictor with configuration
overrides = {
    "conf": 0.5,
    "task": "segment",
    "mode": "predict",
    "model": model_path,
    "quantize": 16,  # Use FP16 for faster inference
    "save": False,
    "imgsz": 1280,
}
predictor = SAM3SemanticPredictor(overrides=overrides)


for img in image_paths:
    print(img_path)
    
    # Query with multiple text prompts
    predictor.set_image(img)
    results = predictor(text=["vegetation"])

    # Save results to disk
    img_name, img_ext = os.path.splitext(
        os.path.basename(result.path)
    )
    filename = os.path.join(
        pred_output_folder,
        f"{img_name}_prediction{img_ext}"
    )
    result.save(filename=filename)

    # TODO also save mask